Menyalakan SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10--KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 15:59:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 15:59:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


A. Menyiapkan Streaming Source

In [3]:
# Membuat skema data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Membuat folder streaming
folder_stream = "tugas10_stream"

shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print("Folder streaming berhasil dibuat.")

# Membaca data secara streaming
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Streaming source berhasil dibuat.")

Folder streaming berhasil dibuat.
Streaming source berhasil dibuat.


B. Agregasi per Cabang

In [4]:
# Menambahkan kolom total harga
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan cabang
agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query pertama
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Agregasi per cabang berhasil dijalankan.")

Agregasi per cabang berhasil dijalankan.


C. Agregasi Berdasarkan Window Waktu

In [5]:
# Membaca ulang data streaming
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

# Menghitung total harga
df_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan berdasarkan window waktu 2 menit
agregasi_waktu = df_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

# Menjalankan streaming query kedua
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Agregasi berdasarkan window waktu berhasil dijalankan.")

Agregasi berdasarkan window waktu berhasil dijalankan.


D. Menguji dengan Data Simulasi

In [8]:
import pandas as pd
import time
from datetime import datetime

In [10]:
# Membuat batch pertama
data1 = pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Cappuccino"],
    "qty": [2, 1, 3],
    "harga_satuan": [15000, 18000, 20000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
})

# Menyimpan data ke folder streaming
data1.to_csv(f"{folder_stream}/batch_1.csv", index=False)

time.sleep(6)

# Menampilkan hasil agregasi
spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           18000|             1|
|   Magelang|           90000|             2|
+-----------+----------------+--------------+



In [11]:
# Membuat batch kedua
data2 = pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Solo", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Hitam", "Kopi Susu", "Latte"],
    "qty": [2, 3, 1],
    "harga_satuan": [12000, 15000, 22000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
})

# Menyimpan data ke folder streaming
data2.to_csv(f"{folder_stream}/batch_2.csv", index=False)

time.sleep(6)

# Menampilkan hasil agregasi terbaru
spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           42000|             2|
| Yogyakarta|           45000|             1|
|   Magelang|          112000|             3|
+-----------+----------------+--------------+



In [12]:
# Membuat batch ketiga
data3 = pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Yogyakarta", "Magelang", "Solo"],
    "nama_menu": ["Americano", "Latte", "Cappuccino"],
    "qty": [2, 2, 3],
    "harga_satuan": [18000, 22000, 20000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
})

# Menyimpan data ke folder streaming
data3.to_csv(f"{folder_stream}/batch_3.csv", index=False)

time.sleep(6)

# Menampilkan hasil agregasi terbaru
spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|          102000|             3|
| Yogyakarta|           81000|             2|
|   Magelang|          156000|             4|
+-----------+----------------+--------------+



E. Menyimpan Data Streaming ke Format Parquet

In [13]:
# Membuat folder output dan checkpoint
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

# Menjalankan streaming untuk menyimpan data ke Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("Streaming Parquet berhasil dijalankan.")

# Membuat dan mengirim batch baru
data4 = pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Yogyakarta"],
    "nama_menu": ["Kopi Susu", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [15000, 18000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 2
})

data4.to_csv(f"{folder_stream}/batch_4.csv", index=False)

# Menunggu Spark memproses data
time.sleep(8)

print("Batch baru berhasil dikirim.")

# Melihat file hasil streaming
!ls tugas10_output

# Membaca data Parquet
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show()

Streaming Parquet berhasil dijalankan.


Batch baru berhasil dikirim.
part-00000-76488651-50fb-46e7-8b6b-27f99ce7a9b7-c000.snappy.parquet
part-00001-25a59b86-368d-4eaa-8e6c-f73783978515-c000.snappy.parquet
part-00002-10d9c12d-87e3-4e5e-b323-c023679c8a3f-c000.snappy.parquet
part-00003-3a8eb932-0b08-44f2-b29f-800200a87cf2-c000.snappy.parquet
_spark_metadata
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|      P007| Yogyakarta| Americano|  2|       18000|2026-11-07 16:21:33|
|      P008|   Magelang|     Latte|  2|       22000|2026-11-07 16:21:33|
|      P009|       Solo|Cappuccino|  3|       20000|2026-11-07 16:21:33|
|      P001|   Magelang| Kopi Susu|  2|       15000|2026-11-07 16:19:41|
|      P002|       Solo| Americano|  1|       18000|2026-11-07 16:19:41|
|      P003|   Magelang|Cappuccino|  3|       20000|2026-11-07 16:19:41|
|      P004|       Solo|Ko

In [ ]:
EXPLORR

1. Menampilkan Data Pesanan Secara Real-Time

In [14]:
# Membuat data pesanan baru
data_eksplorasi1 = pd.DataFrame({
    "pesanan_id": ["P101", "P102", "P103"],
    "nama_cabang": ["Magelang", "Solo", "Yogyakarta"],
    "nama_menu": ["Kopi Susu", "Americano", "Cappuccino"],
    "qty": [2, 3, 1],
    "harga_satuan": [15000, 18000, 20000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
})

# Mengirim data ke folder streaming
data_eksplorasi1.to_csv(
    f"{folder_stream}/eksplorasi_1.csv",
    index=False
)

print("Data pesanan berhasil dikirim.")

Data pesanan berhasil dikirim.


2. Menghitung Total Pendapatan Setiap Cabang

In [15]:
# Menghitung total harga setiap pesanan
df_eksplorasi = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan cabang
hasil_cabang = df_eksplorasi.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming
query_eksplorasi1 = hasil_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("eksplorasi_cabang") \
    .start()

time.sleep(6)

# Menampilkan hasil pendapatan
spark.sql("""
    SELECT *
    FROM eksplorasi_cabang
    ORDER BY total_pendapatan DESC
""").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          216000|             6|
|       Solo|          156000|             4|
| Yogyakarta|          119000|             4|
+-----------+----------------+--------------+



3. Mengelompokkan Pendapatan Berdasarkan Waktu

In [18]:
# Import library
import os
import shutil
import time
import pandas as pd
from datetime import datetime

from pyspark.sql.functions import col, window, sum as spark_sum, count

# Menghentikan query sebelumnya jika masih aktif
if 'query_eksplorasi3' in globals():
    if query_eksplorasi3.isActive:
        query_eksplorasi3.stop()

# Membuat folder khusus eksplorasi waktu
folder_waktu = "eksplorasi_waktu_stream"

shutil.rmtree(folder_waktu, ignore_errors=True)
os.makedirs(folder_waktu, exist_ok=True)

# Membaca data streaming
df_eksplorasi_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_waktu)

# Menghitung total harga pesanan
df_eksplorasi_waktu = df_eksplorasi_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan window 2 menit
hasil_waktu = df_eksplorasi_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming
query_eksplorasi3 = hasil_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("eksplorasi_waktu_baru") \
    .start()

print("Streaming berdasarkan waktu berhasil dijalankan.")

# Membuat data pesanan baru
data_waktu = pd.DataFrame({
    "pesanan_id": ["P201", "P202", "P203", "P204"],
    "nama_cabang": ["Magelang", "Solo", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Latte", "Cappuccino"],
    "qty": [2, 3, 1, 2],
    "harga_satuan": [15000, 18000, 22000, 20000],
    "waktu_pesan": [
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ] * 4
})

# Mengirim data ke folder streaming
data_waktu.to_csv(
    f"{folder_waktu}/batch_waktu.csv",
    index=False
)

print("Data pesanan berhasil dikirim.")

# Menunggu Spark memproses seluruh data
query_eksplorasi3.processAllAvailable()

# Menampilkan hasil pengelompokan waktu
spark.sql("""
    SELECT
        window.start,
        window.end,
        total_pendapatan,
        jumlah_pesanan
    FROM eksplorasi_waktu_baru
""").show(truncate=False)

Streaming berdasarkan waktu berhasil dijalankan.
Data pesanan berhasil dikirim.
+-------------------+-------------------+----------------+--------------+
|start              |end                |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-07 16:30:00|2026-11-07 16:32:00|146000          |4             |
+-------------------+-------------------+----------------+--------------+



4. Menyimpan Data Streaming ke Format Parquet

In [19]:
# Membuat folder output dan checkpoint
folder_output = "eksplorasi_output"
folder_checkpoint = "eksplorasi_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

# Menjalankan streaming ke format Parquet
query_eksplorasi4 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("Proses penyimpanan Parquet dimulai.")

# Mengirim data baru
data_eksplorasi2 = pd.DataFrame({
    "pesanan_id": ["P104", "P105"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Latte", "Kopi Hitam"],
    "qty": [2, 1],
    "harga_satuan": [22000, 12000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 2
})

data_eksplorasi2.to_csv(
    f"{folder_stream}/eksplorasi_2.csv",
    index=False
)

time.sleep(8)

# Melihat file yang berhasil disimpan
!ls eksplorasi_output

# Membaca kembali data Parquet
df_hasil_parquet = spark.read.parquet(folder_output)
df_hasil_parquet.show()

Proses penyimpanan Parquet dimulai.
part-00000-7844de21-c0cd-4201-ab72-bd5fecd3454b-c000.snappy.parquet
part-00000-98f63d36-baa9-4984-aec0-41635a0f1b62-c000.snappy.parquet
part-00001-02a5a861-2011-4c7e-9259-1c1a270e8847-c000.snappy.parquet
part-00002-4c99be4d-b7e4-4894-971d-5974392fa170-c000.snappy.parquet
_spark_metadata
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|      P101|   Magelang| Kopi Susu|  2|       15000|2026-11-07 16:24:40|
|      P102|       Solo| Americano|  3|       18000|2026-11-07 16:24:40|
|      P103| Yogyakarta|Cappuccino|  1|       20000|2026-11-07 16:24:40|
|      P001|   Magelang| Kopi Susu|  2|       15000|2026-11-07 16:19:41|
|      P002|       Solo| Americano|  1|       18000|2026-11-07 16:19:41|
|      P003|   Magelang|Cappuccino|  3|       20000|2026-11-07 16:19:41|
|      P004|       

5. Menghentikan Seluruh Proses Streaming

In [21]:
spark.stop()
print("Seluruh proses eksplorasi berhasil dihentikan.")

Seluruh proses eksplorasi berhasil dihentikan.
